# 潜在変数モデルと混合モデル

観測データだけを見ると一つの集まりに見えても、裏側では複数の生成過程が混ざっていることがあります。潜在変数モデルは、観測されない原因 z を置き、z が x を生むと考えます。混合モデルはその基本形です。どの成分がどの観測を担当したかを確率で推定しながら、成分ごとの分布も同時に学習します。

## 観測されない割当を確率で扱う

混合モデルでは、最初に成分 z を選び、その成分の分布から観測 x が出ます。観測できるのは x だけで、z は見えません。EM アルゴリズムは、見えない z をいきなり一つに決めず、各成分の担当度として持ちます。この担当度を責任率と呼びます。

In [ ]:
import math
import random
from statistics import mean

random.seed(31)

def clamp(p, eps=1e-9):
    return min(max(p, eps), 1 - eps)


def normalize(weights):
    total = sum(weights)
    if total <= 0:
        return [1 / len(weights) for _ in weights]
    return [w / total for w in weights]

## 1ビット観測では成分を同定しにくい

表か裏かだけを観測するコイン投げを考えます。実は2種類のコインが混ざっていても、1回の観測は0か1だけです。この場合、観測分布は結局「1が出る確率」だけに縮約されます。複数成分を置いても、異なるパラメータが同じ観測分布を作れてしまいます。

In [ ]:
def sample_coin_mixture(n, pi, p0, p1):
    xs = []
    zs = []
    for _ in range(n):
        z = 0 if random.random() < pi else 1
        p = p0 if z == 0 else p1
        xs.append(1 if random.random() < p else 0)
        zs.append(z)
    return xs, zs

coin_x, coin_z = sample_coin_mixture(80, pi=0.65, p0=0.85, p1=0.25)
print('heads ratio:', round(sum(coin_x) / len(coin_x), 3))
print('latent counts:', coin_z.count(0), coin_z.count(1))
print('first observations:', coin_x[:24])

1ビット観測では、混合比と2つのコイン確率を別々に確定するだけの情報が足りません。潜在変数モデルは強力ですが、観測に含まれる情報量が少なすぎると、もっともらしい説明が複数残ります。これを同定不能性と呼びます。

In [ ]:
def bernoulli_prob(x, p):
    p = clamp(p)
    return p if x == 1 else 1 - p


def coin_mixture_prob(x, pi, p0, p1):
    return pi * bernoulli_prob(x, p0) + (1 - pi) * bernoulli_prob(x, p1)

candidates = [
    (0.65, 0.85, 0.25),
    (0.50, 0.95, 0.15),
    (0.75, 0.78, 0.08),
]

for params in candidates:
    prob_one = coin_mixture_prob(1, *params)
    ll = sum(math.log(coin_mixture_prob(x, *params)) for x in coin_x)
    print(params, 'P(x=1)=', round(prob_one, 3), 'loglik=', round(ll, 2))

## 多次元にすると成分の形が見える

各観測が6個の0/1値を持つと、前半が1になりやすい成分、後半が1になりやすい成分のような構造を推定できます。成分ごとに確率ベクトルを持つモデルを混合ベルヌーイ分布と呼びます。

In [ ]:
def sample_binary_vectors(n=180):
    pi = 0.58
    mu0 = [0.88, 0.78, 0.72, 0.18, 0.14, 0.10]
    mu1 = [0.16, 0.22, 0.30, 0.82, 0.74, 0.68]
    xs = []
    zs = []
    for _ in range(n):
        z = 0 if random.random() < pi else 1
        mu = mu0 if z == 0 else mu1
        xs.append([1 if random.random() < p else 0 for p in mu])
        zs.append(z)
    return xs, zs

binary_x, binary_z = sample_binary_vectors()
print('first rows:', binary_x[:5])
print('latent counts:', binary_z.count(0), binary_z.count(1))
print('dimension means:', [round(mean(row[j] for row in binary_x), 3) for j in range(6)])

混合ベルヌーイでは、成分 k のパラメータは各次元が1になる確率です。観測 x の確率は、成分ごとの確率を混合比で足し合わせたものになります。責任率は、各観測に対して「成分 k が担当した事後確率」です。

In [ ]:
def bernoulli_vector_prob(x, mu):
    prob = 1.0
    for xi, p in zip(x, mu):
        p = clamp(p)
        prob *= p if xi == 1 else 1 - p
    return prob


def responsibilities_bernoulli_mixture(data, pis, mus):
    resp = []
    for x in data:
        weights = [pis[k] * bernoulli_vector_prob(x, mus[k]) for k in range(len(pis))]
        resp.append(normalize(weights))
    return resp


def loglik_bernoulli_mixture(data, pis, mus):
    total = 0.0
    for x in data:
        p = sum(pis[k] * bernoulli_vector_prob(x, mus[k]) for k in range(len(pis)))
        total += math.log(max(p, 1e-300))
    return total

pis = [0.5, 0.5]
mus = [
    [0.65, 0.60, 0.55, 0.45, 0.40, 0.35],
    [0.35, 0.40, 0.45, 0.55, 0.60, 0.65],
]
for x, r in zip(binary_x[:4], responsibilities_bernoulli_mixture(binary_x[:4], pis, mus)):
    print(x, 'responsibility=', [round(v, 3) for v in r])

## EM は責任率とパラメータ更新を交互に行う

E-step は現在のパラメータで責任率を計算します。M-step は責任率を重みとして、混合比と各成分の確率ベクトルを更新します。硬いクラスタ番号ではなく重み付きの統計量を使うため、境界の観測も自然に扱えます。

In [ ]:
def m_step_bernoulli_mixture(data, resp, alpha=0.5):
    n = len(data)
    d = len(data[0])
    k_count = len(resp[0])
    nk = [sum(r[k] for r in resp) for k in range(k_count)]
    pis = [clamp(nk[k] / n) for k in range(k_count)]
    mus = []
    for k in range(k_count):
        row = []
        for j in range(d):
            numerator = sum(resp[i][k] * data[i][j] for i in range(n)) + alpha
            denominator = nk[k] + 2 * alpha
            row.append(clamp(numerator / denominator))
        mus.append(row)
    return normalize(pis), mus


def run_bernoulli_em(data, n_iter=35, seed=0):
    rng = random.Random(seed)
    d = len(data[0])
    pis = normalize([rng.random() + 0.2 for _ in range(2)])
    mus = [[rng.uniform(0.15, 0.85) for _ in range(d)] for _ in range(2)]
    trace = []
    for _ in range(n_iter):
        resp = responsibilities_bernoulli_mixture(data, pis, mus)
        pis, mus = m_step_bernoulli_mixture(data, resp)
        trace.append(loglik_bernoulli_mixture(data, pis, mus))
    return pis, mus, resp, trace

pis, mus, resp, trace = run_bernoulli_em(binary_x, seed=4)
print('first loglik:', round(trace[0], 2), 'last loglik:', round(trace[-1], 2))
print('pis:', [round(v, 3) for v in pis])
for k, mu in enumerate(mus):
    print('mu', k, [round(v, 3) for v in mu])

尤度は反復で上がります。ただし、成分0と成分1の名前には意味がありません。2つの成分を入れ替えても同じ分布を表せます。これをラベルスイッチングと呼びます。評価するときは、成分番号そのものではなく、分離できている構造を見ます。

In [ ]:
def hard_assign(resp):
    return [max(range(len(r)), key=lambda k: r[k]) for r in resp]


def best_binary_accuracy(pred, true):
    acc = sum(p == t for p, t in zip(pred, true)) / len(true)
    flipped = sum((1 - p) == t for p, t in zip(pred, true)) / len(true)
    return max(acc, flipped)

assign = hard_assign(resp)
print('best assignment accuracy:', round(best_binary_accuracy(assign, binary_z), 3))
print('loglik tail:', [round(v, 2) for v in trace[-5:]])

## 初期値依存と局所解

EM は各反復で尤度を下げにくい一方、出発点によって止まる場所が変わります。成分数が多い、データが重なっている、片方の成分が小さい、といった条件では局所解や成分崩壊が起きやすくなります。複数初期値で走らせ、検証尤度や情報量規準で比較します。

In [ ]:
runs = []
for seed in range(10):
    p, m, r, tr = run_bernoulli_em(binary_x, seed=seed)
    acc = best_binary_accuracy(hard_assign(r), binary_z)
    runs.append((tr[-1], acc, p, m))

runs.sort(key=lambda row: row[0], reverse=True)
for rank, (ll, acc, p, m) in enumerate(runs[:3], start=1):
    print('rank', rank, 'loglik=', round(ll, 2), 'acc=', round(acc, 3), 'pis=', [round(v, 3) for v in p])

## 連続値では混合ガウス分布を使う

連続値の点群では、成分分布としてガウス分布を置くことが多くなります。混合ガウス分布では、各成分が平均と分散を持ちます。E-step は各点の責任率を計算し、M-step は責任率で重み付けして平均と分散を更新します。

In [ ]:
def sample_gmm_2d(n=260):
    params = [
        (0.55, (-1.8, -1.0), (0.45, 0.65)),
        (0.45, (2.0, 1.5), (0.70, 0.50)),
    ]
    xs = []
    zs = []
    for _ in range(n):
        z = 0 if random.random() < params[0][0] else 1
        _, mu, sigma = params[z]
        xs.append((random.gauss(mu[0], sigma[0]), random.gauss(mu[1], sigma[1])))
        zs.append(z)
    return xs, zs

gmm_x, gmm_z = sample_gmm_2d()
print('first points:', [tuple(round(v, 2) for v in x) for x in gmm_x[:5]])
print('latent counts:', gmm_z.count(0), gmm_z.count(1))

対角共分散に限定すると、分散は各軸ごとに持てます。フル共分散より表現力は落ちますが、EM の形を見通しやすく、数値的にも安定させやすい設定です。

In [ ]:
def gaussian_diag_pdf(x, mu, var):
    vx = max(var[0], 1e-6)
    vy = max(var[1], 1e-6)
    dx = (x[0] - mu[0]) ** 2 / vx
    dy = (x[1] - mu[1]) ** 2 / vy
    return math.exp(-0.5 * (dx + dy)) / (2 * math.pi * math.sqrt(vx * vy))


def responsibilities_gmm(data, pis, mus, vars_):
    resp = []
    for x in data:
        weights = [pis[k] * gaussian_diag_pdf(x, mus[k], vars_[k]) for k in range(len(pis))]
        resp.append(normalize(weights))
    return resp


def m_step_gmm(data, resp, min_var=1e-3):
    n = len(data)
    k_count = len(resp[0])
    nk = [sum(r[k] for r in resp) for k in range(k_count)]
    pis = normalize([v / n for v in nk])
    mus = []
    vars_ = []
    for k in range(k_count):
        mx = sum(resp[i][k] * data[i][0] for i in range(n)) / max(nk[k], 1e-12)
        my = sum(resp[i][k] * data[i][1] for i in range(n)) / max(nk[k], 1e-12)
        vx = sum(resp[i][k] * (data[i][0] - mx) ** 2 for i in range(n)) / max(nk[k], 1e-12)
        vy = sum(resp[i][k] * (data[i][1] - my) ** 2 for i in range(n)) / max(nk[k], 1e-12)
        mus.append((mx, my))
        vars_.append((max(vx, min_var), max(vy, min_var)))
    return pis, mus, vars_


def loglik_gmm(data, pis, mus, vars_):
    total = 0.0
    for x in data:
        p = sum(pis[k] * gaussian_diag_pdf(x, mus[k], vars_[k]) for k in range(len(pis)))
        total += math.log(max(p, 1e-300))
    return total

初期値は単純にデータから2点を選びます。実務では k-means 初期化や複数初期値を使います。E-step で所属の確率を更新し、M-step で各成分のパラメータを更新する関係は、混合ベルヌーイと同じです。

In [ ]:
def run_gmm_em(data, n_iter=40, seed=0):
    rng = random.Random(seed)
    mus = [data[rng.randrange(len(data))], data[rng.randrange(len(data))]]
    xs = [x for x, _ in data]
    ys = [y for _, y in data]
    base_var = (max(mean((x - mean(xs)) ** 2 for x in xs), 1e-3), max(mean((y - mean(ys)) ** 2 for y in ys), 1e-3))
    vars_ = [base_var, base_var]
    pis = [0.5, 0.5]
    trace = []
    for _ in range(n_iter):
        resp = responsibilities_gmm(data, pis, mus, vars_)
        pis, mus, vars_ = m_step_gmm(data, resp)
        trace.append(loglik_gmm(data, pis, mus, vars_))
    return pis, mus, vars_, resp, trace

pis_g, mus_g, vars_g, resp_g, trace_g = run_gmm_em(gmm_x, seed=3)
print('first loglik:', round(trace_g[0], 2), 'last loglik:', round(trace_g[-1], 2))
print('pis:', [round(v, 3) for v in pis_g])
for k in range(2):
    print('component', k, 'mu=', tuple(round(v, 3) for v in mus_g[k]), 'var=', tuple(round(v, 3) for v in vars_g[k]))

責任率は境界付近の点で特に意味を持ちます。片方の成分に完全に押し込まず、どちらの成分がどれくらい説明しているかを確率で持つため、重なった分布を扱いやすくなります。

In [ ]:
center = ((mus_g[0][0] + mus_g[1][0]) / 2, (mus_g[0][1] + mus_g[1][1]) / 2)
probes = [
    center,
    mus_g[0],
    mus_g[1],
    (center[0] + 0.5, center[1]),
    (center[0] - 0.5, center[1]),
]
for x in probes:
    r = responsibilities_gmm([x], pis_g, mus_g, vars_g)[0]
    print('x=', tuple(round(v, 2) for v in x), 'responsibility=', [round(v, 3) for v in r])

## 成分数は当てはまりと複雑さの釣り合いで選ぶ

成分数を増やすほど訓練データには合わせやすくなります。しかし、不要な成分は過学習や不安定化を招きます。BIC は、尤度が高いほど良く、パラメータ数が多いほど罰を与える指標です。

In [ ]:
def fit_single_gaussian_diag(data):
    mx = mean(x for x, _ in data)
    my = mean(y for _, y in data)
    vx = mean((x - mx) ** 2 for x, _ in data)
    vy = mean((y - my) ** 2 for _, y in data)
    pis = [1.0]
    mus = [(mx, my)]
    vars_ = [(max(vx, 1e-3), max(vy, 1e-3))]
    return pis, mus, vars_, loglik_gmm(data, pis, mus, vars_)


def bic(loglik, n_samples, n_params):
    return -2 * loglik + n_params * math.log(n_samples)

one_pis, one_mus, one_vars, one_ll = fit_single_gaussian_diag(gmm_x)
two_ll = trace_g[-1]
# 1成分: 平均2 + 分散2 = 4。2成分: 混合比1 + 平均4 + 分散4 = 9。
print('K=1 loglik:', round(one_ll, 2), 'BIC:', round(bic(one_ll, len(gmm_x), 4), 2))
print('K=2 loglik:', round(two_ll, 2), 'BIC:', round(bic(two_ll, len(gmm_x), 9), 2))

混合モデルの中心は、見えない割当を推定しながら分布を学ぶことです。1ビット観測では同定不能性が目立ち、多次元観測では成分の形が見え、連続値では GMM として同じ EM の骨格が使えます。VAE や変分推論へ進むと、潜在変数はより高次元で連続的になり、責任率の代わりに近似事後分布を学ぶ形へ発展します。